# 🐚 Akoya Pearl Miner - Google Colab

**Setup:** Runtime → Change runtime type → GPU (T4)

Jalankan semua cell dari atas ke bawah.

In [ ]:
#@title 1️⃣ Check GPU & Install Tools
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader
!apt-get update -qq && apt-get install -y -qq skopeo umoci > /dev/null 2>&1
print('✅ Tools installed')

In [ ]:
#@title 2️⃣ Download Akoya Miner
import os, shutil

MINER_DIR = '/content/akoya-miner'

# Clean previous
if os.path.exists(MINER_DIR):
    shutil.rmtree(MINER_DIR)
if os.path.exists('/content/akoya-oci'):
    shutil.rmtree('/content/akoya-oci')
if os.path.exists('/content/akoya-bundle'):
    shutil.rmtree('/content/akoya-bundle')

print('Downloading Akoya miner image...')
!skopeo copy docker://registry.akoyapool.com/akoya-miner:latest oci:/content/akoya-oci:latest
print('Unpacking...')
!umoci unpack --image /content/akoya-oci:latest /content/akoya-bundle

# Copy app files
os.makedirs(MINER_DIR, exist_ok=True)
!cp -r /content/akoya-bundle/rootfs/app/* {MINER_DIR}/
!chmod +x {MINER_DIR}/akoya-miner

# Verify
assert os.path.exists(f'{MINER_DIR}/akoya-miner'), '❌ Miner binary not found!'
print(f'\n✅ Miner downloaded to {MINER_DIR}')
!ls {MINER_DIR}/lib/libpearl_gemm_capi_*.so 2>/dev/null && echo '✅ Kernel libs found' || echo '⚠️ No kernel libs'

In [ ]:
#@title 3️⃣ Setup GPU Kernel
import subprocess, os

MINER_DIR = '/content/akoya-miner'
lib_dir = f'{MINER_DIR}/lib'

cc = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip().split('\n')[0]
major, minor = cc.split('.')
print(f'GPU compute: {major}.{minor}')

# Find best kernel
available = [f for f in os.listdir(lib_dir) if 'gemm_capi_' in f]
print(f'Available kernels: {available}')

if int(major) == 12: src = 'blackwell'
elif int(major) == 9: src = 'h100'
elif int(major) == 8 and int(minor) == 9: src = 'ada'
else: src = 'portable'

target = f'{lib_dir}/libpearl_gemm_capi.so'
lib_file = f'{lib_dir}/libpearl_gemm_capi_{src}.so'

if not os.path.exists(lib_file):
    # Fallback to portable
    lib_file = f'{lib_dir}/libpearl_gemm_capi_portable.so'
    src = 'portable'

if os.path.lexists(target):
    os.unlink(target)

if os.path.exists(lib_file):
    os.symlink(lib_file, target)
    print(f'✅ Kernel: {src}')
else:
    print(f'❌ No kernel found! Available: {available}')

In [ ]:
#@title 4️⃣ Test Miner Binary
import subprocess, os

MINER_DIR = '/content/akoya-miner'
os.environ['LD_LIBRARY_PATH'] = f"{MINER_DIR}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')

# Test if binary runs
result = subprocess.run(
    [f'{MINER_DIR}/akoya-miner', '--help'],
    capture_output=True, text=True,
    env=os.environ,
    timeout=10
)
if result.returncode == 0 or 'akoya' in result.stdout.lower() or 'akoya' in result.stderr.lower():
    print('✅ Miner binary works!')
    print(result.stdout[:200] if result.stdout else result.stderr[:200])
else:
    print(f'❌ Miner failed!')
    print(f'stdout: {result.stdout[:500]}')
    print(f'stderr: {result.stderr[:500]}')
    print(f'returncode: {result.returncode}')
    # Check missing libs
    !ldd {MINER_DIR}/akoya-miner 2>&1 | grep 'not found'

In [ ]:
#@title 5️⃣ START MINING 🚀
import subprocess, os, signal, time

MINER_DIR = '/content/akoya-miner'
WALLET = 'prl1pq4v3a3677vymqt47jg6qvcxhgavrldmu4xe2k9hlx95lqqht2fyssutugf'
WORKER = 'colab-t4'

env = os.environ.copy()
env['AKOYA_POOL_WALLET'] = WALLET
env['AKOYA_POOL_WORKER'] = WORKER
env['AKOYA_POOL_HOST'] = 'pool-v2.akoyapool.com'
env['AKOYA_POOL_PORT'] = '443'
env['AKOYA_POOL_USE_TLS'] = '1'
env['AKOYA_GPU_INDICES'] = 'all'
env['AKOYA_METRICS_PORT'] = '9100'
env['AKOYA_PEARL_GEMM_LIB'] = f'{MINER_DIR}/lib/libpearl_gemm_capi.so'
env['AKOYA_PEARL_MINING_LIB'] = f'{MINER_DIR}/lib/libpearl_mining_capi.so'
env['LD_LIBRARY_PATH'] = f"{MINER_DIR}/lib:" + env.get('LD_LIBRARY_PATH', '')

os.makedirs('/var/lib/akoya-miner', exist_ok=True)

print(f'🐚 Starting Akoya Pearl Miner')
print(f'   Wallet: {WALLET[:15]}...{WALLET[-8:]}')
print(f'   Worker: {WORKER}')
print(f'   Pool: pool-v2.akoyapool.com:443')
print('='*60)

proc = subprocess.Popen(
    [f'{MINER_DIR}/akoya-miner', 'mine-blocks'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    start = time.time()
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line.rstrip(), flush=True)
    # If process exited immediately
    remaining = proc.stdout.read()
    if remaining:
        print(remaining)
    if proc.returncode != 0:
        print(f'\n❌ Miner exited with code {proc.returncode}')
        print('Try: check GPU kernel compatibility or run Cell 4 for diagnostics')
except KeyboardInterrupt:
    print('\n⏹️ Stopping miner...')
    proc.send_signal(signal.SIGINT)
    proc.wait(timeout=30)
    print(f'✅ Miner stopped. Ran {int(time.time()-start)}s')